Import all needed libraries

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers

!Check installed libraies

In [ ]:
%who

Import excel file with data

In [ ]:
df = pd.read_excel(r"path")

Check basic information about the data

divide data in train and test sets

In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['variables'])
y = df.into_default
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.3,random_state=42)

process the variables in the dataset

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. labels 
num_labels = X_train.select_dtypes('number').columns
cat_labels = X_train.select_dtypes('object').columns

# 2. instantiate preprocessors
num_preprocessor = StandardScaler()
cat_preprocessor = OneHotEncoder(drop = 'if_binary')

# 3. combine both preprocessors into one
preprocessor = ColumnTransformer([('cat',cat_preprocessor,cat_labels),
                                  ('num',num_preprocessor,num_labels)])

preprocessor.fit(X_train)

feature_labels = preprocessor.transformers_[0][1].get_feature_names_out().tolist()
feature_labels.extend(num_labels)

X_train_prep = preprocessor.transform(X_train)
X_test_prep = preprocessor.transform(X_test)

X_test_prep = X_test_prep.toarray()
X_train_prep = X_train_prep.toarray()

X_train_prep = pd.DataFrame(X_train_prep,columns = feature_labels)
X_test_prep = pd.DataFrame(X_test_prep,columns = feature_labels)



Define parameters for neural network

In [ ]:
from keras import regularizers

def model_init():
    input_layer = layers.Input(shape = (X_train_prep.shape[1],), name = 'input')
    
    h1 = layers.Dense(128, activation = "relu", name = 'h1', kernel_regularizer=regularizers.L1L2(0.0001))(input_layer)
    h2= layers.Dropout(0.3)(h1)
    h3 = layers.Dense(64, activation = "relu",  name = 'h2', kernel_regularizer=regularizers.L1L2(0.0001))(h2)
    h4 =layers.Dropout(0.3)(h3)
    
    output_layer = layers.Dense(1,activation = "sigmoid", name = 'out')(h4)#relu??? as target cannot be negative
    
    model = keras.Model(input_layer, output_layer)
    
    model.compile(loss = 'binary_crossentropy', optimizer = 'adam', metrics = ['AUC','F1Score', 'Precision', 'Recall', 'FalseNegatives', 'FalsePositives', 'TruePositives', 'TrueNegatives'])###think about loss (mse is good for regression) and optimizer
    return model


In [ ]:
model = model_init()

In [ ]:
model.summary()

In [ ]:
import datetime
logs_path = "path"+ datetime.datetime.now().strftime("%H-%M-%S")


callbacks = [
  tf.keras.callbacks.EarlyStopping(patience=40, monitor='val_Recall'),#can tweak patience
  tf.keras.callbacks.TensorBoard(log_dir=logs_path),#monitors in real time
  tf.keras.callbacks.ReduceLROnPlateau(monitor='val_Recall', factor=.5,patience=10)
  ]



model.fit(X_train_prep,
          y_train,
          batch_size = 128, #why???
          epochs = 1000,
          validation_split=.2,
          callbacks=callbacks,
          class_weight={0:1,1:7.6}
          )
          

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score

In [ ]:
y_train_pred = model.predict(X_train_prep)
y_train_pred = pd.DataFrame(y_train_pred)
y_train_pred['default'] = y_train_pred[0].apply(lambda x: 1 if x > 0.55 else 0)
print("AUC: " + str(roc_auc_score(y_train,y_train_pred[0])) + ", F1Score: " + str(f1_score(y_train,y_train_pred['default'])) + ", Precision: " + str(precision_score(y_train,y_train_pred['default'])) + ", Recall: " + str(recall_score(y_train,y_train_pred['default'])))

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_train,y_train_pred['default'])

In [ ]:
y_test_pred = model.predict(X_test_prep)
y_test_pred = pd.DataFrame(y_test_pred)
y_test_pred['default'] = y_test_pred[0].apply(lambda x: 1 if x > 0.55 else 0)
y_test_pred
print("AUC: " + str(roc_auc_score(y_test,y_test_pred[0])) + ", F1Score: " + str(f1_score(y_test,y_test_pred['default'])) + ", Precision: " + str(precision_score(y_test,y_test_pred['default'])) + ", Recall: " + str(recall_score(y_test,y_test_pred['default'])))

In [ ]:
confusion_matrix(y_test,y_test_pred['default'])

In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_test, y_test_pred[0])
auc_score = roc_auc_score(y_test, y_test_pred[0])

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc_score:.3f}')
plt.plot([0, 1], [0, 1], 'k--', label='Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_train, y_train_pred[0])
auc_score = roc_auc_score(y_train, y_train_pred[0])

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {auc_score:.3f}')
plt.plot([0, 1], [0, 1], 'k--', label='Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
!pip install nnv
from nnv import NNV

layersList = [
    {"title":"input\n 54", "units": 54, "color": "darkBlue"},
    {"title":"hidden 1\n 128 (relu)", "units": 128},
    {"title":"hidden 2\n 64 (relu)", "units": 64, "edges_color":"red", "edges_width":2},
    {"title":"output\n(sigmoid)", "units": 1,"color": "darkBlue"},
]

NNV(layersList).render(save_to_file="my_example.png")